In [4]:
# =========================
# INSTALL
# =========================
!pip install fvcore av pytorchvideo peft

# =========================
# IMPORTS
# =========================
import os
import cv2
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

from peft import LoraConfig, get_peft_model

# =========================
# CONFIG
# =========================
DATA_DIR = "/kaggle/input/datasets/sabinasabitovaseds/wlasl100/WLASL_100/"
BATCH_SIZE = 8
LR = 1e-4
EPOCHS = 20
SAVE_PATH = "best_slowfast_lora.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# DATASET
# =========================
class WLASLDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root_dir = os.path.join(root_dir, mode)
        self.mode = mode
        
        self.classes = sorted(os.listdir(self.root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(self.root_dir, cls)
            for vid in os.listdir(cls_path):
                if vid.endswith('.mp4'):
                    self.samples.append((os.path.join(cls_path, vid), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        window_size = 64
        
        if self.mode == 'train':
            start = np.random.randint(0, max(1, total_frames - window_size))
        else:
            start = max(0, (total_frames - window_size) // 2)

        buffer = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

        for _ in range(window_size):
            ret, frame = cap.read()
            if not ret:
                frame = buffer[-1] if buffer else np.zeros((224,224,3), np.uint8)
            else:
                frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),(224,224))
            buffer.append(frame)

        cap.release()
        return np.array(buffer)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        raw_clip = self._get_frames(video_path)

        raw_clip = raw_clip.astype(np.float32)/255.0
        raw_clip = (raw_clip - 0.45)/0.225

        slow = torch.from_numpy(raw_clip[np.arange(8)*8]).permute(3,0,1,2)
        fast = torch.from_numpy(raw_clip[np.arange(32)*2]).permute(3,0,1,2)

        return [slow, fast], label


def build_model(num_classes=100):
    model = torch.hub.load(
        'facebookresearch/pytorchvideo',
        'slowfast_r50',
        pretrained=True
    )

    # Replace classifier
    in_features = model.blocks[6].proj.in_features
    model.blocks[6].proj = nn.Linear(in_features, num_classes)

   
    for p in model.parameters():
        p.requires_grad = False

   
    for p in model.blocks[5].parameters():
        p.requires_grad = True

    for p in model.blocks[6].parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.2f}%)")

    return model

# =========================
# LOADERS
# =========================
train_loader = DataLoader(WLASLDataset(DATA_DIR,'train'), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(WLASLDataset(DATA_DIR,'val'),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(WLASLDataset(DATA_DIR,'test'),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =========================
# TRAIN SETUP
# =========================
model = build_model().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

best_val = 0

# =========================
# TRAIN LOOP
# =========================
for epoch in range(EPOCHS):

    model.train()
    train_loss, train_correct, total = 0,0,0

    for inputs, labels in train_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)

        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (out.argmax(1)==labels).sum().item()
        total += labels.size(0)

    # VALIDATION
    model.eval()
    val_correct, val_total = 0,0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [i.to(device) for i in inputs]
            labels = labels.to(device)

            out = model(inputs)
            val_correct += (out.argmax(1)==labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), SAVE_PATH)

    print(f"Epoch {epoch+1}: Train Acc {100*train_correct/total:.2f} | Val Acc {val_acc:.2f}")

# =========================
# TEST (SINGLE CLIP)
# =========================
print("\nTesting (single center clip)...")

model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

correct, total = 0,0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)

        out = model(inputs)

        correct += (out.argmax(1)==labels).sum().item()
        total += labels.size(0)

print(f"Final Test Accuracy: {100*correct/total:.2f}%")

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main


Trainable params: 230500 / 33874988 (0.68%)
Epoch 1: Train Acc 1.39 | Val Acc 2.37
Epoch 2: Train Acc 3.40 | Val Acc 4.73
Epoch 3: Train Acc 5.48 | Val Acc 5.92
Epoch 4: Train Acc 9.08 | Val Acc 6.51
Epoch 5: Train Acc 9.08 | Val Acc 6.21
Epoch 6: Train Acc 11.30 | Val Acc 7.10
Epoch 7: Train Acc 14.56 | Val Acc 9.76
Epoch 8: Train Acc 17.61 | Val Acc 9.76
Epoch 9: Train Acc 18.38 | Val Acc 11.24
Epoch 10: Train Acc 22.40 | Val Acc 11.24
Epoch 11: Train Acc 22.54 | Val Acc 11.54
Epoch 12: Train Acc 25.38 | Val Acc 13.02
Epoch 13: Train Acc 24.69 | Val Acc 13.91
Epoch 14: Train Acc 28.02 | Val Acc 14.50
Epoch 15: Train Acc 29.13 | Val Acc 13.91
Epoch 16: Train Acc 31.97 | Val Acc 15.09
Epoch 17: Train Acc 31.35 | Val Acc 14.20
Epoch 18: Train Acc 33.63 | Val Acc 15.38
Epoch 19: Train Acc 32.45 | Val Acc 14.79
Epoch 20: Train Acc 35.44 | Val Acc 15.68

Testing (single center clip)...
Final Test Accuracy: 10.08%
